In [89]:
import pandas as pd
import openpyxl
import ifcopenshell
import ifcopenshell.util.element
from collections import Counter
from sklearn.model_selection import train_test_split


In [90]:
df_matrix = pd.read_csv(r"C:\Users\lucas.galicioli\Desktop\ifc-classifier\notebooks\data\interim\cls_matrix_v2.csv")
df_matrix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311763 entries, 0 to 311762
Data columns (total 16 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Class                        311763 non-null  object 
 1   PredefinedType               232368 non-null  object 
 2   BuildingStorey               305822 non-null  object 
 3   Material                     168265 non-null  object 
 4   Name                         311247 non-null  object 
 5   PSET_RÔGGA.RÔGGA_SEÇÃO       135208 non-null  object 
 6   PSET_RÔGGA.RÔGGA_DESCRIÇÃO   203962 non-null  object 
 7   Width                        29264 non-null   float64
 8   Thickness                    0 non-null       float64
 9   Length                       109955 non-null  float64
 10  Height                       27144 non-null   float64
 11  FileName                     311763 non-null  object 
 12  GlobalId                     311763 non-null  object 
 13 

C:\Users\lucas.galicioli\AppData\Local\Temp\ipykernel_34712\2983744699.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_matrix = pd.read_csv(r"C:\Users\lucas.galicioli\Desktop\ifc-classifier\notebooks\data\interim\cls_matrix_v2.csv")


In [91]:
df_matrix.describe()

,Width,Thickness,Length,Height
count,29264.000000,0.0,109955.000000,27144.000000
mean,3.796640,NaN,104.987219,29.103547
std,25.206752,NaN,327.553395,50.815407
min,0.020000,NaN,0.005027,0.080000
25%,0.080000,NaN,6.196000,10.000000
50%,0.200000,NaN,31.902008,10.000000
75%,0.200000,NaN,115.103288,20.000000
max,1012.638805,NaN,10244.091941,374.580000


In [92]:
df_matrix.isnull().sum()

Class                               0
PredefinedType                  79395
BuildingStorey                   5941
Material                       143498
Name                              516
PSET_RÔGGA.RÔGGA_SEÇÃO         176555
PSET_RÔGGA.RÔGGA_DESCRIÇÃO     107801
Width                          282499
Thickness                      311763
Length                         201808
Height                         284619
FileName                            0
GlobalId                            0
GUID                                0
Ô_CLS_DISCIPLINAS                   0
Ô_CLS_CLASSIFICAÇÃO_SOLIBRI         0
dtype: int64

In [93]:
import numpy as np # Importante ter o numpy

# Lista das colunas de texto que queremos limpar
colunas_para_limpar = [
    'Material',
    'PSET_RÔGGA.RÔGGA_SEÇÃO',
    'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
    'Ô_CLS_DISCIPLINAS',
    'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI'
]

# Valor que representa "ausente" no seu arquivo
valor_ausente = 'NaN'

# Novo valor que vamos colocar no lugar
novo_valor = 'Desconhecido'

# Este loop vai passar por cada coluna da lista e fazer a substituição
for coluna in colunas_para_limpar:
    print(f"Limpando a coluna: {coluna}...")
    df_matrix[coluna] = df_matrix[coluna].replace(valor_ausente, novo_valor)

print("\nLimpeza concluída!")

Limpando a coluna: Material...
Limpando a coluna: PSET_RÔGGA.RÔGGA_SEÇÃO...
Limpando a coluna: PSET_RÔGGA.RÔGGA_DESCRIÇÃO...
Limpando a coluna: Ô_CLS_DISCIPLINAS...
Limpando a coluna: Ô_CLS_CLASSIFICAÇÃO_SOLIBRI...

Limpeza concluída!


In [94]:
display(df_matrix['Material'].value_counts())

Material
Concreto C40                                                     19608
AÇO DIN 2440                                                     16738
Linha Corrugado PVC Reforçado Laranja Antichamas                 13234
<Unnamed>                                                        10837
PVC Marrom                                                        9234
                                                                 ...  
Piso emborrachado azul (grânulo de pneu pigmentado) - Aubicon        1
Piso:ROGGA_PISO_COMUM_ARGAMASSA+CERÂMICA-5cm                         1
wire_196088225                                                       1
wire_228153184                                                       1
 FIBRA ÓTICA                                                         1
Name: count, Length: 346, dtype: int64

In [95]:
# Célula 7 (COMO ESTAVA ANTES)

features_selecionadas = [
    'Class', 'PredefinedType', 'BuildingStorey', 'Ô_CLS_DISCIPLINAS', 
    'Material', 'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
]
X = df_matrix[features_selecionadas].copy()
print(f"DataFrame 'X' criado com {len(features_selecionadas)} features.")

DataFrame 'X' criado com 7 features.


In [96]:
# NOVA CÉLULA (vamos chamar de Célula 7.5) - CORRIGIDA

print("Aplicando Frequency Encoding (com tratamento CORRETO de NaNs)...")

colunas_para_freq_encoding = [
    'Material',
    'PSET_RÔGGA.RÔGGA_SEÇÃO',
    'PSET_RÔGGA.RÔGGA_DESCRIÇÃO'
]

# Vamos usar o mesmo placeholder da Célula 5
placeholder = "Desconhecido" 

for col in colunas_para_freq_encoding:
    if col in X.columns:
        
        # PASSO 1 (A CORREÇÃO): Preenche NaNs REAIS (np.nan) com o placeholder
        # Agora "Desconhecido" será uma categoria válida.
        X[col] = X[col].fillna(placeholder)
        
        # PASSO 2: Calcula a frequência. Agora "Desconhecido" será
        # a categoria mais frequente para Material e PSET_SEÇÃO.
        frequencias = X[col].value_counts(normalize=True)
        
        # PASSO 3: Mapeia a frequência de volta para a coluna
        X[col + '_Freq'] = X[col].map(frequencias)
        
        # PASSO 4: Remove a coluna de texto original
        X = X.drop(col, axis=1)
        
        print(f"-> Coluna '{col}' substituída por '{col}_Freq'.")
        # Vamos imprimir a frequência do placeholder para confirmar
        print(f"   Frequência de '{placeholder}': {frequencias.get(placeholder, 0):.2%}")

print("\nFrequency Encoding (corrigido) concluído.")
print("Novas colunas em X:", X.columns.tolist())
print("\nPróximo passo: Execute a Célula 8 (One-Hot Encoding).")

Aplicando Frequency Encoding (com tratamento CORRETO de NaNs)...
-> Coluna 'Material' substituída por 'Material_Freq'.
   Frequência de 'Desconhecido': 46.03%
-> Coluna 'PSET_RÔGGA.RÔGGA_SEÇÃO' substituída por 'PSET_RÔGGA.RÔGGA_SEÇÃO_Freq'.
   Frequência de 'Desconhecido': 56.63%
-> Coluna 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO' substituída por 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO_Freq'.
   Frequência de 'Desconhecido': 34.58%

Frequency Encoding (corrigido) concluído.
Novas colunas em X: ['Class', 'PredefinedType', 'BuildingStorey', 'Ô_CLS_DISCIPLINAS', 'Material_Freq', 'PSET_RÔGGA.RÔGGA_SEÇÃO_Freq', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO_Freq']

Próximo passo: Execute a Célula 8 (One-Hot Encoding).


In [97]:
# Aplica o One-Hot Encoding em todas as colunas de texto dentro de X
X_encoded = pd.get_dummies(X)

print("One-Hot Encoding concluído!")

One-Hot Encoding concluído!


In [98]:
# 1. Mostra as dimensões (linhas, colunas) do novo DataFrame.
# O número de colunas vai aumentar BASTANTE!
print(X_encoded.shape)

# 2. Mostra as 5 primeiras linhas do novo DataFrame para vermos a estrutura
print(X_encoded.head())

(311763, 312)
   Material_Freq  PSET_RÔGGA.RÔGGA_SEÇÃO_Freq  \
0       0.000013                     0.566312   
1       0.000013                     0.566312   
2       0.000013                     0.566312   
3       0.001309                     0.566312   
4       0.001309                     0.566312   

   PSET_RÔGGA.RÔGGA_DESCRIÇÃO_Freq  Class_IfcAirTerminal  Class_IfcAlarm  ...  \
0                         0.345779                 False           False  ...   
1                         0.345779                 False           False  ...   
2                         0.345779                 False           False  ...   
3                         0.000016                 False           False  ...   
4                         0.000141                 False           False  ...   

   Ô_CLS_DISCIPLINAS_Pressurização  \
0                            False   
1                            False   
2                            False   
3                            False   
4             

In [99]:
# # Lista das colunas numéricas
# colunas_numericas = ['Width', 'Thickness', 'Length', 'Height']

# # colunas_numericas = ['Width', 'Length']

# # Este loop passa por cada coluna da lista
# for coluna in colunas_numericas:
#     # 1. Calcula a mediana da coluna
#     mediana = X_encoded[coluna].median()
    
#     # 2. Usa .fillna() para preencher os NaN com o valor da mediana
#     X_encoded[coluna] = X_encoded[coluna].fillna(mediana)
    
#     print(f"Valores nulos na coluna '{coluna}' preenchidos com a mediana ({mediana}).")

# print("\nPreenchimento de dados numéricos concluído!")

In [100]:
# Soma todos os valores nulos em todas as colunas do DataFrame
total_nulos = X_encoded.isnull().sum().sum()

print(f"\nTotal de valores nulos no DataFrame final 'X_encoded': {total_nulos}")

if total_nulos == 0:
    print("Parabéns! Seus dados estão 100% limpos e prontos para o treinamento do modelo!")
else:
    print("Ainda existem valores nulos. Precisamos investigar o que aconteceu.")


Total de valores nulos no DataFrame final 'X_encoded': 0
Parabéns! Seus dados estão 100% limpos e prontos para o treinamento do modelo!


In [101]:
from sklearn.preprocessing import LabelEncoder

# 1. Seleciona a primeira coluna que queremos prever do DataFrame original
y1 = df_matrix['Ô_CLS_CLASSIFICAÇÃO_SOLIBRI']

# 2. Cria uma instância do codificador
le1 = LabelEncoder()

# 3.Ajusta o codificador aos seus dados e os transforma em números
y1_encoded = le1.fit_transform(y1)

print("Variável alvo 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI' foi codificada com sucesso!")
print("Exemplo dos primeiros 5 valores codificados:", y1_encoded[:5])


Variável alvo 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI' foi codificada com sucesso!
Exemplo dos primeiros 5 valores codificados: [137 137 137  56  56]


In [102]:
# Certifique-se de que o numpy está importado (geralmente como np)
import numpy as np
print("\nPasso 2.5 de 3: Removendo classes com poucas amostras...")

# 1. Encontre as classes únicas e suas contagens
classes, class_counts = np.unique(y1_encoded, return_counts=True)

# 2. Identifique as classes que têm 3 OU MAIS amostras
#    (Esta é a correção para a validação cruzada)
min_amostras = 3 
valid_classes = classes[class_counts >= min_amostras]

# 3. Crie a máscara
mask = np.isin(y1_encoded, valid_classes)

# 4. Filtre X_encoded e y1_encoded
X_encoded_filtered = X_encoded[mask]
y1_encoded_filtered = y1_encoded[mask]

print(f"--> Filtro aplicado: Mínimo de {min_amostras} amostras por classe.")
print(f"--> Amostras removidas: {len(y1_encoded) - len(y1_encoded_filtered)}")
print(f"--> Shape de X após a filtragem: {X_encoded_filtered.shape}")
print(f"--> Shape de y após a filtragem: {y1_encoded_filtered.shape}")

# ------------------------------------------------------------------------------

# AGORA, no Passo 3, use os dataframes filtrados:
print("\nPasso 3 de 3: Dividindo os dados em conjuntos de treino e teste...")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_filtered,   # Use a versão filtrada
    y1_encoded_filtered,  # Use a versão filtrada
    test_size=0.25,
    random_state=42,
    stratify=y1_encoded_filtered # Estratifique pelo y filtrado
)

print("--> Dados divididos com sucesso!")
print(f"--> Shape do X_train final: {X_train.shape}")
print(f"--> Shape do y_train final: {y_train.shape}")
print("\n✅ PREPARAÇÃO CONCLUÍDA.")


Passo 2.5 de 3: Removendo classes com poucas amostras...
--> Filtro aplicado: Mínimo de 3 amostras por classe.
--> Amostras removidas: 8
--> Shape de X após a filtragem: (311755, 312)
--> Shape de y após a filtragem: (311755,)

Passo 3 de 3: Dividindo os dados em conjuntos de treino e teste...
--> Dados divididos com sucesso!
--> Shape do X_train final: (233816, 312)
--> Shape do y_train final: (233816,)

✅ PREPARAÇÃO CONCLUÍDA.


In [103]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder # <-- Importe o LabelEncoder aqui

# ==============================================================================
# BLOCO DE CÓDIGO CONSOLIDADO (COM A CORREÇÃO)
# ==============================================================================

print("Passo 1 de 4: Aplicando One-Hot Encoding em X...")
# X e y1_encoded vêm da Célula 13
X_encoded = pd.get_dummies(X)
X_encoded.columns = X_encoded.columns.str.replace(r'\\[|\\]|<', '_', regex=True)
print(f"--> DataFrame 'X_encoded' criado com shape: {X_encoded.shape}")

# ------------------------------------------------------------------------------

print("\nPasso 2 de 4: Removendo classes com 1 amostra...")
# y1_encoded e le1 (o encoder "master") vêm da Célula 13

classes, class_counts = np.unique(y1_encoded, return_counts=True)
valid_classes = classes[class_counts >= 2]
mask = np.isin(y1_encoded, valid_classes)

X_encoded_filtered = X_encoded[mask]
y1_encoded_filtered = y1_encoded[mask] # <-- Este 'y' tem "buracos" (ex: 0, 1, 3, 4)

print(f"--> Amostras removidas: {len(y1_encoded) - len(y1_encoded_filtered)}")
print(f"--> Shape de X após a filtragem: {X_encoded_filtered.shape}")

# ------------------------------------------------------------------------------

print("\nPasso 3 de 4: Re-codificando 'y' para XGBoost (Removendo Gaps)...")
# ESTA É A CORREÇÃO CRUCIAL
# Nós criamos um NOVO encoder (le_xgb) que transforma os rótulos com "buracos"
# (ex: [0, 1, 3, 4]) em um rótulo contínuo que o XGBoost aceita (ex: [0, 1, 2, 3]).

le_xgb = LabelEncoder()
y_xgb_encoded = le_xgb.fit_transform(y1_encoded_filtered)

print(f"--> 'y' re-codificado. Novo range de classes: 0 a {y_xgb_encoded.max()}")
print(f"--> Total de classes únicas para o modelo: {len(le_xgb.classes_)}")

# ------------------------------------------------------------------------------

print("\nPasso 4 de 4: Dividindo os dados em conjuntos de treino e teste...")
# AGORA, usamos o X filtrado e o NOVO y_xgb_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_filtered,   # <-- Use o X filtrado
    y_xgb_encoded,        # <-- Use o NOVO 'y' re-codificado
    test_size=0.25,
    random_state=42,
    stratify=y_xgb_encoded  # <-- Estratifique pelo NOVO 'y'
)

print("--> Dados divididos com sucesso!")
print(f"--> Shape do X_train final: {X_train.shape}")
print(f"--> Shape do y_train final: {y_train.shape}")
print("\n✅ PREPARAÇÃO CONCLUÍDA. Agora você pode rodar a célula de treinamento.")
# ==============================================================================

Passo 1 de 4: Aplicando One-Hot Encoding em X...
--> DataFrame 'X_encoded' criado com shape: (311763, 312)

Passo 2 de 4: Removendo classes com 1 amostra...
--> Amostras removidas: 2
--> Shape de X após a filtragem: (311761, 312)

Passo 3 de 4: Re-codificando 'y' para XGBoost (Removendo Gaps)...
--> 'y' re-codificado. Novo range de classes: 0 a 139
--> Total de classes únicas para o modelo: 140

Passo 4 de 4: Dividindo os dados em conjuntos de treino e teste...
--> Dados divididos com sucesso!
--> Shape do X_train final: (233820, 312)
--> Shape do y_train final: (233820,)

✅ PREPARAÇÃO CONCLUÍDA. Agora você pode rodar a célula de treinamento.


In [104]:
# Célula 15 (MODIFICADA com Early Stopping - A Abordagem Correta)

import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ==============================================================================
# BLOCO DE TREINAMENTO COM EARLY STOPPING
# ==============================================================================

print("Iniciando o treinamento do XGBoost com Early Stopping...")

# 1. Defina o classificador XGBoost com parâmetros para Early Stopping
#    (Assumindo que 'le_xgb' da Célula 14 está na memória)
try:
    total_classes = len(le_xgb.classes_)
    print(f"Detectado o número total de classes: {total_classes}")
except NameError:
    print("ERRO: A variável 'le_xgb' da Célula 14 não foi encontrada. Rode a Célula 14 primeiro.")
    total_classes = 140 # Fallback

xgb_classifier = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=total_classes,  # Importante: Informa o total de classes
    
    tree_method="hist", 
    device="cuda",
    use_label_encoder=False,
    
    # --- Parâmetros para Otimização ---
    n_estimators=1000,        # Um número ALTO. O Early Stopping encontrará o ideal.
    learning_rate=0.05,       # Uma taxa de aprendizado baixa (mais preciso)
    max_depth=7,              # Árvores um pouco mais profundas
    subsample=0.8,
    colsample_bytree=0.8,
    
    # Habilita a parada antecipada
    early_stopping_rounds=10, # Para se a pontuação não melhorar por 10 rodadas
    
    eval_metric='mlogloss'
)

# 2. Treine o modelo
#    Nós passamos o (X_test, y_test) como o "eval_set"
#    O modelo vai usá-lo para decidir quando parar.
print("Treinando e monitorando a performance no set de teste...")
xgb_classifier.fit(
    X_train, 
    y_train,
    eval_set=[(X_test, y_test)], # <-- A MÁGICA ACONTECE AQUI
    verbose=50 # Imprime o progresso a cada 50 árvores
)

print("\nTreinamento concluído!")
print(f"Melhor iteração (nº de árvores): {xgb_classifier.best_iteration}")

# ------------------------------------------------------------------------------

# 3. Usa o modelo treinado para fazer previsões
#    (O XGBoost já usa o "best_iteration" automaticamente)
print("\nRealizando previsões nos dados de teste com o modelo otimizado...")
y_pred = xgb_classifier.predict(X_test)

# 4. Avalia a acurácia do modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAcurácia FINAL no conjunto de teste: {accuracy:.4f}")
print(f"Isso significa que o modelo acertou {accuracy:.2%} das classificações!")

# 5. Gera um relatório de classificação detalhado
# (O código de correção dos labels das Células 15/16 permanece o mesmo)
try:
    original_gappy_labels = le_xgb.classes_
    target_names_corretos = le1.inverse_transform(original_gappy_labels)
    target_names_corretos = target_names_corretos.astype(str)
    labels_para_report = np.arange(len(target_names_corretos))

    print("\nRelatório de Classificação Detalhado:")
    print(classification_report(
        y_test, 
        y_pred, 
        labels=labels_para_report,
        target_names=target_names_corretos, 
        zero_division=0
    ))
except Exception as e:
    print(f"\nErro ao gerar relatório de classificação: {e}")
# ==============================================================================

Iniciando o treinamento do XGBoost com Early Stopping...
Detectado o número total de classes: 140
Treinando e monitorando a performance no set de teste...


c:\Users\lucas.galicioli\Desktop\ifc-classifier\.venv\Lib\site-packages\xgboost\callback.py:386: UserWarning: [15:03:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	validation_0-mlogloss:2.10663
[50]	validation_0-mlogloss:0.18859
[100]	validation_0-mlogloss:0.06661
[150]	validation_0-mlogloss:0.05259
[200]	validation_0-mlogloss:0.05008
[250]	validation_0-mlogloss:0.04931
[300]	validation_0-mlogloss:0.04902
[350]	validation_0-mlogloss:0.04885
[400]	validation_0-mlogloss:0.04874
[450]	validation_0-mlogloss:0.04866
[500]	validation_0-mlogloss:0.04859
[541]	validation_0-mlogloss:0.04856

Treinamento concluído!
Melhor iteração (nº de árvores): 532

Realizando previsões nos dados de teste com o modelo otimizado...

Acurácia FINAL no conjunto de teste: 0.9788
Isso significa que o modelo acertou 97.88% das classificações!

Relatório de Classificação Detalhado:
                                                                                                    precision    recall  f1-score   support

                                                                                    Aba de Fachada       0.96      0.77      0.86        35
               

In [105]:
# # Célula 15 (MODIFICADA com 'sample_weight' para Desbalanceamento)

# import xgboost as xgb
# from sklearn.metrics import accuracy_score, classification_report
# import numpy as np
# # IMPORTANTE: Importe o 'class_weight'
# from sklearn.utils.class_weight import compute_sample_weight

# # ==============================================================================
# # BLOCO DE TREINAMENTO COM EARLY STOPPING E PESOS DE CLASSE
# # ==============================================================================

# print("Iniciando o treinamento do XGBoost com Early Stopping e Pesos de Amostra...")

# # 1. Defina o classificador XGBoost (como antes)
# try:
#     total_classes = len(le_xgb.classes_)
#     print(f"Detectado o número total de classes: {total_classes}")
# except NameError:
#     print("ERRO: A variável 'le_xgb' da Célula 14 não foi encontrada. Rode a Célula 14 primeiro.")
#     total_classes = 140 

# xgb_classifier = xgb.XGBClassifier(
#     objective='multi:softmax',
#     num_class=total_classes,  
#     tree_method="hist", 
#     device="cuda",
#     use_label_encoder=False,
#     n_estimators=1000,        
#     learning_rate=0.05,       
#     max_depth=7,              
#     subsample=0.8,
#     colsample_bytree=0.8,
#     early_stopping_rounds=10, 
#     eval_metric='mlogloss'
# )

# # 2. CALCULAR OS PESOS DE AMOSTRA (A NOVA ETAPA)
# #    Isso vai dar um "peso" maior para amostras de classes raras
# print("Calculando pesos de amostra para lidar com classes desbalanceadas...")
# # 'balanced' calcula automaticamente o peso inverso da frequência da classe
# sample_weights = compute_sample_weight(
#     class_weight='balanced',
#     y=y_train
# )
# print("Pesos de amostra calculados.")

# # 3. Treine o modelo (PASSANDO OS PESOS)
# print("Treinando e monitorando a performance no set de teste...")
# xgb_classifier.fit(
#     X_train, 
#     y_train,
#     sample_weight=sample_weights, # <-- A MÁGICA ACONTECE AQUI
#     eval_set=[(X_test, y_test)], 
#     verbose=50 
# )

# print("\nTreinamento concluído!")
# print(f"Melhor iteração (nº de árvores): {xgb_classifier.best_iteration}")

# # ------------------------------------------------------------------------------

# # 4. Usa o modelo treinado para fazer previsões
# print("\nRealizando previsões nos dados de teste com o modelo otimizado...")
# y_pred = xgb_classifier.predict(X_test)

# # 5. Avalia a acurácia do modelo
# accuracy = accuracy_score(y_test, y_pred)
# print(f"\nAcurácia FINAL no conjunto de teste: {accuracy:.4f}")
# print(f"Isso significa que o modelo acertou {accuracy:.2%} das classificações!")

# # 6. Gera um relatório de classificação detalhado
# try:
#     original_gappy_labels = le_xgb.classes_
#     target_names_corretos = le1.inverse_transform(original_gappy_labels)
#     target_names_corretos = target_names_corretos.astype(str)
#     labels_para_report = np.arange(len(target_names_corretos))

#     print("\nRelatório de Classificação Detalhado:")
#     print(classification_report(
#         y_test, 
#         y_pred, 
#         labels=labels_para_report,
#         target_names=target_names_corretos, 
#         zero_division=0
#     ))
# except Exception as e:
#     print(f"\nErro ao gerar relatório de classificação: {e}")
# # ==============================================================================

In [106]:
# ==============================================================================
# NOVO BLOCO: EXIBINDO OS RESULTADOS DETALHADOS (REAL vs. PREVISTO)
# ==============================================================================
print("\n" + "="*80)
print("VERIFICAÇÃO DETALHADA DAS PREVISÕES (REAL vs. PREVISTO)")
print("="*80)

try:
    # 1. Importar Pandas para criar a tabela de visualização
    import pandas as pd
    pd.set_option('display.max_rows', 200) # Para mostrar mais linhas se necessário
    pd.set_option('display.max_columns', 10) # Para mostrar mais colunas

    # 2. Criar um mapa do ID numérico (0, 1, 2...) para o Nome da classe
    #    labels_para_report = [0, 1, 2, ..., 124]
    #    target_names_corretos = ["NomeA", "NomeB", "NomeC", ..., "NomeZ"]
    mapa_id_para_nome = dict(zip(labels_para_report, target_names_corretos))

    # 3. Criar o DataFrame principal de comparação
    df_comparacao = pd.DataFrame({
        'Real_ID': y_test,
        'Previsto_ID': y_pred
    })

    # 4. Mapear os IDs de volta para os nomes originais para melhor leitura
    df_comparacao['Real_Nome'] = df_comparacao['Real_ID'].map(mapa_id_para_nome)
    df_comparacao['Previsto_Nome'] = df_comparacao['Previsto_ID'].map(mapa_id_para_nome)

    # 5. Adicionar uma coluna para ver facilmente os acertos/erros
    df_comparacao['Acertou?'] = df_comparacao['Real_ID'] == df_comparacao['Previsto_ID']

    # 6. Mostrar as primeiras 25 previsões
    print("\n[--- Amostra das Previsões (primeiras 25) ---]")
    
    # Tenta usar display() se estiver em um notebook (Jupyter, Colab)
    # Senão, usa print() normal, que formata bem o DataFrame.
    try:
        from IPython.display import display
        display(df_comparacao.head(25))
    except ImportError:
        # .to_string() garante uma boa formatação no console padrão
        print(df_comparacao.head(25).to_string()) 

    # 7. Mostrar APENAS os erros (se houver)
    #    (No seu caso, com 100% de acurácia, esta tabela estará vazia)
    df_erros = df_comparacao[df_comparacao['Acertou?'] == False]
    
    if df_erros.empty:
        print("\n[--- Análise de Erros ---]")
        print("Parabéns! O modelo não cometeu erros no conjunto de teste.")
    else:
        print(f"\n[--- Exibindo os {len(df_erros)} Erros de Classificação ---]")
        try:
            from IPython.display import display
            display(df_erros)
        except ImportError:
            print(df_erros.to_string())

except ImportError:
    # Fallback caso o usuário não tenha pandas instalado
    print("\nPara uma visualização detalhada em tabela, instale a biblioteca pandas:")
    print("pip install pandas")
    print("\nResultados brutos (sem pandas - 20 primeiras amostras):")
    print(f"Valores Reais (y_test):    {y_test[:20]}...")
    print(f"Valores Previstos (y_pred): {y_pred[:20]}...")
# ==============================================================================


VERIFICAÇÃO DETALHADA DAS PREVISÕES (REAL vs. PREVISTO)

[--- Amostra das Previsões (primeiras 25) ---]


,Real_ID,Previsto_ID,Real_Nome,Previsto_Nome,Acertou?
0,88,89,Paredes Externas,"Paredes Internas, Divisórias",False
1,98,98,"Portas Internas, Externas","Portas Internas, Externas",True
2,105,105,Ralos Sifonados,Ralos Sifonados,True
3,35,35,Conexões,Conexões,True
4,35,35,Conexões,Conexões,True
5,123,123,Tubulação,Tubulação,True
6,35,35,Conexões,Conexões,True
7,139,139,Viga,Viga,True
8,37,37,Conexões da Linha Frigorígena,Conexões da Linha Frigorígena,True
9,35,35,Conexões,Conexões,True



[--- Exibindo os 1653 Erros de Classificação ---]


,Real_ID,Previsto_ID,Real_Nome,Previsto_Nome,Acertou?
0,88,89,Paredes Externas,"Paredes Internas, Divisórias",False
19,135,131,Tubulação de Água Pluvial,Tubulação de Ventilação,False
133,135,131,Tubulação de Água Pluvial,Tubulação de Ventilação,False
159,110,111,Revestimento Externo de Parede,Revestimento Interno de Parede,False
206,136,89,Unclassified,"Paredes Internas, Divisórias",False
...,...,...,...,...,...
77554,84,85,NC – Sifão para Cozinha,NC – Torneiras e registros,False
77573,46,47,Dutos de Ventilação e Exaustão (Central e prum...,Dutos de Ventilação e Exaustão (Distribuição),False
77745,78,77,Módulo de interruptor convencional,Módulo de força,False
77884,136,67,Unclassified,"Janelas internas, Janelas Externas, Venezianas",False


In [107]:
import joblib

# Salva o modelo treinado em um arquivo
joblib.dump(xgb_classifier, 'ifc_classifier_disciplinas_v1.pkl')

# Salva também o LabelEncoder, pois você precisará dele para decodificar as previsões
joblib.dump(le1, 'label_encoder_disciplinas_v1.pkl')

print("Modelo e LabelEncoder salvos com sucesso!")

Modelo e LabelEncoder salvos com sucesso!


In [108]:
import json

# --- SALVAR ARTEFATOS ADICIONAIS ---

# 3. Salvar a lista de colunas do modelo
colunas_do_modelo = X_encoded.columns.tolist()
with open('colunas_modelo_disciplinas.json', 'w') as f:
    json.dump(colunas_do_modelo, f)

# 4. Salvar as medianas de treinamento
colunas_numericas = ['Width', 'Thickness', 'Length', 'Height']
medianas = df_matrix[colunas_numericas].median().to_dict()
with open('medianas_treinamento.json', 'w') as f:
    json.dump(medianas, f)

print("Artefatos de pré-processamento (colunas e medianas) salvos com sucesso!")

Artefatos de pré-processamento (colunas e medianas) salvos com sucesso!


In [109]:
# import ifcopenshell
# import ifcopenshell.util.element
# import pandas as pd
# import os

# # --- Caminho do arquivo IFC ---
# ifc_file_path = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-PCI-PR-0001-BIM-EMB-GER SPK-R03.ifc"

# # --- Funções Auxiliares (com a função de quantidade modificada) ---

# def get_building_storey(element):
#     """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
#     try:
#         spatial_container = ifcopenshell.util.element.get_container(element)
#         if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
#             return spatial_container.Name
#     except Exception:
#         pass
#     return None

# def get_material_name(element):
#     """ Extrai o nome do material associado ao elemento. """
#     material = ifcopenshell.util.element.get_material(element)
#     if not material:
#         return None
#     if hasattr(material, 'Name'):
#         return material.Name
#     elif hasattr(material, 'MaterialLayers'):
#         layer_names = [
#             layer.Material.Name 
#             for layer in material.MaterialLayers 
#             if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
#         ]
#         return ', '.join(layer_names) if layer_names else None
#     return None
    
# # --- SOLUÇÃO 2: Função de quantidade compatível com versões antigas ---
# def get_quantity_value_legacy(element, quantity_name):
#     """
#     Busca por uma quantidade específica (ex: 'Width') e retorna seu valor.
#     Esta versão é compatível com versões mais antigas do ifcopenshell sem 'get_qsets'.
#     """
#     # Itera através das relações de definição do elemento
#     for definition in getattr(element, 'IsDefinedBy', []):
#         if definition.is_a('IfcRelDefinesByProperties'):
#             prop_set = definition.RelatingPropertyDefinition
#             # Verifica se é um conjunto de quantidades (IfcElementQuantity)
#             if prop_set.is_a('IfcElementQuantity'):
#                 # Itera através das quantidades dentro do conjunto
#                 for quantity in prop_set.Quantities:
#                     if quantity.Name == quantity_name:
#                         # Extrai o valor do atributo correto (ex: LengthValue, AreaValue)
#                         value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
#                         if value_attribute:
#                             return getattr(quantity, value_attribute)
#     return None

# # --- Processamento Principal ---

# try:
#     ifc_file = ifcopenshell.open(ifc_file_path)
#     file_name = os.path.basename(ifc_file_path)

#     element_data = []
#     products = ifc_file.by_type('IfcProduct')

#     print(f"Processando {len(products)} elementos do arquivo: {file_name}...")

#     for product in products:
#         if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
#             continue

#         psets = ifcopenshell.util.element.get_psets(product)
#         rogga_pset = psets.get('PSET_RÔGGA', {})

#         element_info = {
#             'GlobalId': product.GlobalId,
#             'FileName': file_name,
#             'Class': product.is_a(),
#             'PredefinedType': getattr(product, 'PredefinedType', None),
#             'Name': getattr(product, 'Name', None),
#             'BuildingStorey': get_building_storey(product),
#             'Material': get_material_name(product),
#             'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
#             'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
            
#             # ATENÇÃO: Usando a nova função 'legacy' aqui
#             'Width': get_quantity_value_legacy(product, 'Width'),
#             'Thickness': get_quantity_value_legacy(product, 'Thickness'),
#             'Length': get_quantity_value_legacy(product, 'Length'),
#             'Height': get_quantity_value_legacy(product, 'Height'),
#         }
        
#         element_data.append(element_info)

#     df_ifc_data = pd.DataFrame(element_data)

#     desired_order = [
#         'Class', 'PredefinedType', 'BuildingStorey', 'Material', 'Name',
#         'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
#         'Width', 'Thickness', 'Length', 'Height',
#         'FileName', 'GlobalId'
#     ]

#     for col in desired_order:
#         if col not in df_ifc_data.columns:
#             df_ifc_data[col] = None
            
#     df_ifc_data = df_ifc_data[desired_order]

#     print("\nDataset criado com sucesso! Amostra dos dados:")
#     display(df_ifc_data.head().fillna(''))

# except FileNotFoundError:
#     print(f"ERRO: O arquivo não foi encontrado em: {ifc_file_path}")
# except Exception as e:
#     print(f"Ocorreu um erro inesperado: {e}")